In [ ]:
"""
Student Exam Score Prediction
COMP6577001 - Machine Learning Final Project
BINUS University 2025/2026

Regression Only Version
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR


# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "../outputs"
MODEL_DIR = "../models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]

sns.set_theme(style="whitegrid", palette=PALETTE)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11
})


# =============================================================================
# 1. LOAD DATA
# =============================================================================

print("=" * 60)
print("1. LOADING DATASET")
print("=" * 60)

df_raw = pd.read_csv("../data/students_data.csv")

print(f"Raw shape : {df_raw.shape}")

# Remove unused columns
columns_to_drop = []

if "access_to_resources" in df_raw.columns:
    columns_to_drop.append("access_to_resources")

if "student_id" in df_raw.columns:
    columns_to_drop.append("student_id")

if "dropout_risk" in df_raw.columns:
    columns_to_drop.append("dropout_risk")

df = df_raw.drop(columns=columns_to_drop)

print(f"Clean shape : {df.shape}")
print(f"Missing values : {df.isnull().sum().sum()}")


# =============================================================================
# 2. EDA
# =============================================================================

print("\n" + "=" * 60)
print("2. EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# -------------------------------------------------------------------------
# Figure 1 - Exam Score Distribution
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    df["exam_score"],
    bins=30,
    color=PALETTE[1],
    edgecolor="white"
)

mean_score = df["exam_score"].mean()

ax.axvline(
    mean_score,
    color=PALETTE[3],
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_score:.2f}"
)

ax.set_title("Exam Score Distribution", fontweight="bold")
ax.set_xlabel("Exam Score")
ax.set_ylabel("Count")
ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig1_exam_score_distribution.png")
plt.close()

print("Saved fig1_exam_score_distribution.png")


# -------------------------------------------------------------------------
# Figure 2 - Correlation Heatmap
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 10))

corr = df.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax
)

ax.set_title("Feature Correlation Matrix", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig2_correlation_heatmap.png")
plt.close()

print("Saved fig2_correlation_heatmap.png")


# -------------------------------------------------------------------------
# Figure 3 - Key Features vs Exam Score
# -------------------------------------------------------------------------

candidate_features = [
    "study_hours_per_day",
    "attendance_percentage",
    "previous_gpa",
    "mental_health_rating",
    "stress_level"
]

key_features = [f for f in candidate_features if f in df.columns]

fig, axes = plt.subplots(
    1,
    len(key_features),
    figsize=(4 * len(key_features), 4)
)

if len(key_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, key_features):

    ax.scatter(
        df[feat],
        df["exam_score"],
        alpha=0.35,
        s=10
    )

    z = np.polyfit(df[feat], df["exam_score"], 1)
    p = np.poly1d(z)

    xs = np.linspace(
        df[feat].min(),
        df[feat].max(),
        200
    )

    ax.plot(xs, p(xs), linewidth=2)

    corr_val = df[feat].corr(df["exam_score"])

    ax.set_title(
        f"{feat}\nr={corr_val:.2f}",
        fontsize=9,
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig3_features_vs_score.png")
plt.close()

print("Saved fig3_features_vs_score.png")


# =============================================================================
# 3. FEATURE PREPARATION
# =============================================================================

print("\n" + "=" * 60)
print("3. FEATURE ENGINEERING")
print("=" * 60)

FEATURES = [c for c in df.columns if c != "exam_score"]

X = df[FEATURES]
y = df["exam_score"]

print(f"Features used : {len(FEATURES)}")
print(f"Dataset shape : {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")


# =============================================================================
# 4. REGRESSION MODELS
# =============================================================================

print("\n" + "=" * 60)
print("4. EXAM SCORE REGRESSION")
print("=" * 60)

reg_models = {
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0))
    ]),

    "Lasso Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", Lasso(alpha=0.5, max_iter=5000))
    ]),

    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", DecisionTreeRegressor(
            max_depth=6,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", RandomForestRegressor(
            n_estimators=150,
            max_depth=8,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", SVR(
            kernel="rbf",
            C=10,
            epsilon=0.5
        ))
    ])
}

results = {}

for name, model in reg_models.items():

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "predictions": preds,
        "model": model
    }

    print(
        f"{name:20s} | "
        f"RMSE={rmse:.4f} | "
        f"MAE={mae:.4f} | "
        f"R²={r2:.4f}"
    )

best_model_name = max(results, key=lambda x: results[x]["r2"])
best_result = results[best_model_name]

print(f"\nBest model: {best_model_name}")
print(f"R² = {best_result['r2']:.4f}")


# =============================================================================
# 5. VISUALIZATION OF RESULTS
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Model comparison
names = list(results.keys())
scores = [results[n]["r2"] for n in names]

axes[0].barh(names, scores)

axes[0].set_title("Model Comparison (R²)")
axes[0].set_xlabel("R²")

# Actual vs Predicted
preds = best_result["predictions"]

axes[1].scatter(
    y_test,
    preds,
    alpha=0.4
)

lims = [
    min(y_test.min(), preds.min()),
    max(y_test.max(), preds.max())
]

axes[1].plot(lims, lims, "r--")

axes[1].set_title(f"Actual vs Predicted\n{best_model_name}")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

# Residuals
residuals = y_test - preds

axes[2].scatter(
    preds,
    residuals,
    alpha=0.4
)

axes[2].axhline(
    0,
    color="red",
    linestyle="--"
)

axes[2].set_title("Residual Plot")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Residual")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig4_regression_results.png")
plt.close()

print("Saved fig4_regression_results.png")


# =============================================================================
# 6. SAVE MODEL
# =============================================================================

print("\n" + "=" * 60)
print("5. SAVING MODEL")
print("=" * 60)

joblib.dump(
    best_result["model"],
    f"{MODEL_DIR}/exam_score_regressor.pkl"
)

joblib.dump(
    FEATURES,
    f"{MODEL_DIR}/features.pkl"
)

print(f"Saved exam_score_regressor.pkl ({best_model_name})")
print("Saved features.pkl")


# =============================================================================
# 7. FINAL REPORT
# =============================================================================

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(f"Best Regressor : {best_model_name}")
print(f"R²   : {best_result['r2']:.4f}")
print(f"RMSE : {best_result['rmse']:.4f}")
print(f"MAE  : {best_result['mae']:.4f}")

print("\nDone.")
print("Outputs saved to:", OUTPUT_DIR)
print("Model saved to:", MODEL_DIR)

1. LOADING DATASET
Raw shape : (10000, 21)
Clean shape : (10000, 18)
Missing values : 0

2. EXPLORATORY DATA ANALYSIS
Saved fig1_exam_score_distribution.png
Saved fig2_correlation_heatmap.png
Saved fig3_features_vs_score.png

3. FEATURE ENGINEERING
Features used : 17
Dataset shape : (10000, 17)
Training samples : 8000
Testing samples  : 2000

4. EXAM SCORE REGRESSION
Ridge Regression     | RMSE=4.1257 | MAE=3.1164 | R²=0.8697
Lasso Regression     | RMSE=4.1547 | MAE=3.1954 | R²=0.8679
Decision Tree        | RMSE=4.2256 | MAE=3.2224 | R²=0.8633
Random Forest        | RMSE=4.1261 | MAE=3.1672 | R²=0.8697
SVR                  | RMSE=4.4328 | MAE=3.3165 | R²=0.8496

Best model: Ridge Regression
R² = 0.8697
Saved fig4_regression_results.png

5. SAVING MODEL
Saved exam_score_regressor.pkl (Ridge Regression)
Saved features.pkl

FINAL RESULTS
Best Regressor : Ridge Regression
R²   : 0.8697
RMSE : 4.1257
MAE  : 3.1164

Done.
Outputs saved to: ../outputs
Model saved to: ../models
